# C-test ONE · Evaluator

*Ctest-Evaluator-v1.32.0*

Administers the items to the examinees, showing each the prefix and asking for the
next five terms, and scores the replies with a judge model.

| reads | writes |
|---|---|
| `ctest_battery.json`, `ctest_metrics.json` | `ctest_results.json` |

**Needs** a Gemini API key, and `OPENAI_API_KEY` and `CLAUDE_API_KEY` for those providers'
examinees. API examinees need no GPU; local examinees (`LOCAL_EXAMINEES`) do. Examinees and their
token budgets are listed in `API_EXAMINEES` and `LOCAL_EXAMINEES`; always give the budget, since
a bare name takes `DEFAULT_MAX_OUT`.

## Running it

**Locally.** From the repository root, install the requirements and open the notebook in Jupyter:

```
pip install -r requirements.txt
jupyter lab notebooks/
```

API keys are read from environment variables (see `.env.example`). On the first local run the
notebook copies `data/` into a `work/` folder and runs there, so the released data are never
modified; set `CTEST_WORK_DIR` to use another folder.

**Colab.** Upload the data files to `MyDrive/ctest` and mount Drive; keys go in the Secrets panel.

**Kaggle.** Attach a dataset holding the data files and set `KAGGLE_DATASET` to its name; keys go in
Add-ons, Secrets. With Kaggle credentials the notebook can also push its results back as a new
version of the dataset.


In [ ]:
# Local runs work in a copy of the data, so that data/ stays exactly as released.
# On Colab and Kaggle this cell does nothing.
import os, glob, shutil
if not (os.path.exists("/kaggle") or os.path.exists("/content")):
    _root = os.getcwd()
    if os.path.basename(_root) == "notebooks":
        _root = os.path.dirname(_root)
    try:
        from dotenv import load_dotenv          # optional: keys from a .env file
        load_dotenv(os.path.join(_root, ".env"))
    except ImportError:
        pass
    WORK_DIR = os.path.abspath(os.environ.get("CTEST_WORK_DIR", os.path.join(_root, "work")))
    os.makedirs(WORK_DIR, exist_ok=True)
    for _f in glob.glob(os.path.join(_root, "data", "*")):
        _dst = os.path.join(WORK_DIR, os.path.basename(_f))
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst)
    os.chdir(WORK_DIR)
    print(f"local run: working in {WORK_DIR}")


## 1 · Install

In [ ]:
# ---------------------------------------------------------------------------
# If the key cannot be found in the secrets panel, put it here. The environment
# is checked before any provider, so this always wins.
#
# Kaggle secrets are PER NOTEBOOK: having one in another session does not carry
# over. Add-ons -> Secrets, and switch the toggle on for this notebook.
API_KEY_HERE = ""       # e.g. "AIza..."  Leave empty to use the secrets panel.
# ---------------------------------------------------------------------------
import os
if API_KEY_HERE:
    os.environ["GOOGLE_API_KEY"] = API_KEY_HERE
    print("GOOGLE_API_KEY taken from API_KEY_HERE in this cell")

%pip install -q -U google-genai openai anthropic
# transformers and bitsandbytes are for a LOCAL examinee loaded in 4-bit, which
# is how the reference machines run. Kaggle does not ship bitsandbytes, and
# BitsAndBytesConfig imports from transformers whether or not it is installed,
# so a missing library only announces itself when the weights are loaded.
%pip install -q -U transformers accelerate bitsandbytes

import os
try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as _e:
    print("not on Colab, or Drive not mounted:", _e)


## 1b · Preamble

In [ ]:
import os, sys, re, json, time, glob, math, shutil, random, hashlib, platform
import collections, statistics as st, datetime, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

BUILD = "Ctest-Evaluator-v1.32.0"
print("=" * 72)
print(f"{BUILD}")
print("=" * 72)

# The filesystem decides, not an import. Kaggle ships packages that make
# `import google.colab` succeed, which sent ENV to "colab" on Kaggle and killed
# the run on the unmounted-Drive guard.
if os.path.exists("/kaggle"):
    ENV = "kaggle"
elif os.path.isdir("/content"):
    ENV = "colab"
else:
    ENV = "local"
# Two separate places, because they are separate things and always were.
DRIVE_FOLDER   = "ctest"        # Colab: /content/drive/MyDrive/ctest
KAGGLE_DATASET = "ctest-one-data"    # Kaggle: the dataset to read from and push to.
                                # A Kaggle dataset is a FOLDER of files, not a
                                # single file.

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
# OUTDIR is a second place to copy finished files to. On Colab that is Drive.
# On Kaggle there is no such place: an attached dataset is READ-ONLY at
# /kaggle/input, and the working directory is the only writable one and is
# already where files are written. Copying a file onto itself raises
# SameFileError, so OUTDIR stays "." and persistence is the dataset push in the
# cell below.
if ENV == "colab" and os.path.isdir("/content/drive/MyDrive"):
    OUTDIR = DRIVE
else:
    OUTDIR = "."
print(f"environment      {ENV}   python {platform.python_version()}")

# shutil.copy writes a FILE when its destination is not an existing directory, so
# without this every save lands in a file named after the folder and the results
# leave with the session.
if OUTDIR != ".":
    if os.path.exists(OUTDIR) and not os.path.isdir(OUTDIR):
        raise SystemExit(f"{OUTDIR} exists but is a file. Rename or delete it.")
    os.makedirs(OUTDIR, exist_ok=True)
elif ENV == "colab":
    raise SystemExit("Drive is not mounted, so anything written would be lost. Run\n"
                     "    from google.colab import drive; drive.mount('/content/drive')")
if ENV == "kaggle":
    print(f"outputs go to    the working directory, then pushed to the "
          f"{KAGGLE_DATASET} dataset")
    print( "                 (a dataset cannot be written to in place; see the "
           "persistence cell)")
if ENV != "kaggle":
    print(f"outputs go to    {OUTDIR}")

try:
    import torch
    if torch.cuda.is_available():
        print("GPU              attached, and not needed. Switch to CPU and save quota.")
except Exception:
    pass

def _secret(n):
    """Look for a secret, and SAY what each provider reported.

    An earlier version swallowed the Kaggle error and fell through to the Colab
    branch, which raises on Kaggle, so the result was a bare None and nothing
    said why. Kaggle secrets are PER NOTEBOOK: one attached to another session
    does not carry over."""
    if os.environ.get(n):
        return os.environ[n]
    notes = []
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(n)
        if v:
            return v
        notes.append("kaggle: returned nothing")
    except ImportError:
        pass                                  # not on Kaggle, which is fine
    except Exception as e:
        notes.append(f"kaggle: {type(e).__name__}: {str(e)[:90]}")
    try:
        from google.colab import userdata
        v = userdata.get(n)
        if v:
            return v
        notes.append("colab: returned nothing")
    except ImportError:
        pass
    except Exception as e:
        notes.append(f"colab: {type(e).__name__}: {str(e)[:90]}")
    for m in notes:
        print(f"                 {m}")
    return None

_key = _secret("GOOGLE_API_KEY")
if _key: os.environ["GOOGLE_API_KEY"] = _key
print(f"GOOGLE_API_KEY   {'found' if _key else 'MISSING'}")

# The dataset, from a secret when there is one, so two accounts can work in
# parallel on separate copies with this notebook byte-identical in both.
_ds = _secret("KAGGLE_DATASET")
if _ds:
    KAGGLE_DATASET = _ds.strip()
print(f"KAGGLE_DATASET   {KAGGLE_DATASET}"
      + ("   from the secret" if _ds else "   the built-in default"))

# The other houses. A Kaggle secret is not an environment variable until
# something fetches it, exactly as with the Hub token below, so an examinee
# from OpenAI or Anthropic fails with a bare KeyError unless this runs. The
# names come from KEY_FOR in the composer cell, so a secret with a different
# name is one edit there and none here.
for _kind, _var in (("openai", "OPENAI_API_KEY"),
                    ("anthropic", "CLAUDE_API_KEY")):
    _v = _secret(_var)
    if _v:
        os.environ[_var] = _v
    print(f"{_var:18s} "
          + (f"found, {len(_v)} chars" if _v else
             "ABSENT   <- an examinee from " + _kind + " will fail"))

# The Hub token has to be put INTO the environment before transformers is asked
# for a model. A Kaggle secret is not an environment variable: it lives behind
# UserSecretsClient and is invisible to huggingface_hub until something fetches
# it, which is why a notebook with HF_TOKEN in its secrets still reports
# "unauthenticated requests to the HF Hub". Unauthenticated still works for a
# public model; it is rate limited and slower, and it fails outright on a gated
# one.
#
# Both names are set. huggingface_hub reads HF_TOKEN, and some versions of
# transformers and datasets still look for HUGGING_FACE_HUB_TOKEN.
_hf = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN") \
      or _secret("HUGGINGFACE_TOKEN")
if _hf:
    os.environ["HF_TOKEN"] = _hf
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = _hf
    print(f"HF_TOKEN         found, {len(_hf)} chars, set for the Hub")
else:
    print( "HF_TOKEN         not found. Downloads are unauthenticated: rate")
    print( "                 limited, and gated models will not resolve. On")
    print( "                 Kaggle add it under Add-ons -> Secrets with the")
    print( "                 toggle ON for this notebook; on Colab use the key")
    print( "                 panel. The secret must be named HF_TOKEN.")
if not _key:
    print()
    print("=" * 72)
    print("STOPPING: no GOOGLE_API_KEY, and the composer is an API model")
    print("=" * 72)
    if ENV == "kaggle":
        print("  Kaggle secrets are PER NOTEBOOK. Having one in another session")
        print("  does not carry over: Add-ons -> Secrets, and switch the toggle")
        print("  ON for this notebook. Any lines above say what the lookup")
        print("  reported.")
    elif ENV == "colab":
        print("  Colab: the key panel on the left, and the notebook-access")
        print("  toggle has to be on for this notebook.")
    print()
    print("  Or set API_KEY_HERE in the install cell, which is checked first.")
    raise SystemExit("no GOOGLE_API_KEY")

_PRINT_LOCK = threading.Lock()

def log(*a, **kw):
    with _PRINT_LOCK:
        print(*a, flush=True, **kw)

def stage(t):
    with _PRINT_LOCK:
        print(f"\n{'='*72}\n== {t}\n{'='*72}", flush=True)

def find(f):
    for p in [f, f"/content/{f}", f"{DRIVE}/{f}"] + glob.glob(f"/kaggle/input/**/{f}", recursive=True):
        if os.path.exists(p): return p
    return None

def load(name, default):
    p = find(name)
    if not p:
        log(f"  {name}: absent, starting empty"); return default
    log(f"  {name}: loaded from {p}")
    return json.load(open(p))


## 2 · The run

In [ ]:
# ============================== THE RUN ====================================
# Administers the battery to examinees and scores the answers.
#
# Reads   ctest_battery.json   the items, and n_M, which fixes the prefix
#         ctest_metrics.json   optional: h, so the report can relate score to
#                              difficulty. Absent is fine, scores are still
#                              written.
# Writes  ctest_results.json   one row per (item, examinee, repetition)
#
# It never touches the battery, the pricing or the metrics, so it cannot
# undo an annotation run. But a Kaggle PUSH replaces the whole dataset, and
# every session stages files it merely read, so two sessions pushing at once
# still lose one another's work. That is what PUSH_RESULTS is for.

PUSH_RESULTS   = True   # on Kaggle, push the results back to the dataset as they are written

# ---- who sits the test ----------------------------------------------------
PUSH_MIDRUN    = True   # push to the dataset at every mid-run save, not
API_EXAMINEES = [
    ("gpt-5.6-sol", 16000),
    ("claude-opus-5-fallback", 16000),
    ("claude-sonnet-5-fallback", 16000),
    ("claude-fable-5-1-fallback", 16000),
    ("claude-haiku-4-5-20251001", 8000),
    ("gpt-5-nano", 8000),
    ("gemini-3.8-flash", 8000),
    ("gemini-3.6-flash", 8000),
    ("gemini-3.5-flash-lite", 8000),
    ("gemini-3.1-flash-lite", 8000),
]

LOCAL_EXAMINEES = [
    ("allenai/OLMo-2-1124-7B-Instruct", 2048),
    ("Qwen/Qwen2.5-1.5B-Instruct", 2048),
]    # the two local examinees of the paper. The reference machines sitting their
     # own battery (Appendix E) are a separate run, not part of a standard one.

# A WARNING, not a stop: there may be a reason to repeat, such as measuring
# the judge. But it should not happen by default.
if LOCAL_EXAMINEES and LOCAL_REPS > 1:
    print("")
    print("  " + "!" * 70)
    print(f"  !  LOCAL_REPS = {LOCAL_REPS}. The local path decodes greedily, so every")
    print(f"  !  attempt after the first repeats the same reply, and this run will")
    print(f"  !  take {LOCAL_REPS} times as long for no new information about the model.")
    print(f"  !  Set LOCAL_REPS = 1 unless the point is to measure the judge.")
    print("  " + "!" * 70)

LOCAL_4BIT  = True    # as the reference machines are loaded
LOCAL_THINK = True    # as they run: the tag says 4bit-think

DEFAULT_MAX_OUT = 8000    # the default, for a bare name. Every examinee in this
                          # run has its budget written out.
WALL_CLOCK_MIN  = 690     # stop cleanly before a 12-hour session ends; 0 for none

EXAMINEE_PROBE = True     # ask every examinee one trivial question before
PROBE_RETRIES   = 3     # a busy provider is retried this many times before
                        # it is called dead
PROBE_BACKOFF_S = 20    # seconds between those tries
                          # anything is spent, and STOP if one cannot answer.
                          # A retired or misnamed model returns nothing, and
                          # 162 empty replies score 0% and look like a result.

# Server-side fallbacks are asked for by NAMING the examinee with a
# "-fallback" suffix, not by a switch: "claude-opus-5-fallback" is the same
# model with fallbacks requested, and it is scored as its own examinee beside
# the plain one, so both can be kept and compared.
#
# Refusals cost this battery real signal. On the v17 items opus refused ten and
# sonnet nine, every one on the reversed-word sequences, which read like
# obfuscation and are not, and each refusal is scored wrong. But a fallback
# answer is not the model's own answer, so the two are different examinees
# rather than two attempts at one. Every row records which it was.

# Open-weight examinees, loaded one at a time. EMPTY unless a GPU run is
# wanted: each costs a download and several minutes, and the reference machine
# has to be out of memory first.

# e.g. ["Qwen/Qwen2.5-1.5B-Instruct", "tiiuae/Falcon3-3B-Instruct"]

JUDGE = "gemini-3.6-flash"

# ATTEMPTS PER ITEM, set separately for the two paths, because they are not
# equally random.
#
# An API examinee is asked at EXAM_TEMP = 0 and still varies. Some ignore the
# setting outright (the Claude models report temp_applied = False) and the
# rest vary regardless: over the battery, the share of items given the same
# answer on every attempt is 23 per cent for claude-haiku, 30 for
# gemini-3.5-flash-lite, 45 for gpt-5-nano and 60 for gpt-5.6-sol. For these,
# three attempts measure something one attempt cannot.
REPS = 3

LOCAL_REPS = 1            # decoding is greedy, so repeated attempts are identical;
                          # the released results have three identical attempts each
EXAM_TEMP = 0.0
K_ASK = 5
ONLY_CERTIFIED = True     # administer only items that certified. A
                          # questionable item has a rival that produces the
                          # same prefix, so a wrong answer may be right.

VALID_ONLY = True
CERTIFIED_ANY_MACHINE = True
FILL_GAPS_ONLY = True     # ask only what is missing, so an interrupted run resumes
ITEM_ONLY = []            # empty: every item in the pool
MAX_ITEMS = 0             # 0 for all of them
ITEM_ORDER = "easiest"    # "spread" puts the items in farthest-point order
                          # over (r_hat, t_hat, log K), so a run stopped early
                          # still covers the grid and the cost range. "id" for
                          # the plain order. "easiest" is least expensive
                          # first, for an examinee that may not finish.

# ---- files -------------------------------------------------------------
BATTERY      = "ctest_battery.json"
METRICS      = "ctest_metrics.json"
RESULTS_FILE = "ctest_results.json"
MACHINE_ID  = "qwen3-14b-4bit-think"
MACHINE_TAG = "qwen3-14b"

# ---- carried from the generator, because the shared API cell uses them -
COMPOSER     = "gemini-3.6-flash"
MAX_WORKERS  = 6
DISABLE_AFC = True
TOKEN_BUDGET, MAX_SPEND_USD = 150_000_000, 120.0
USD_PER_MTOK_IN, USD_PER_MTOK_OUT = 0.30, 3.75
API_RETRIES, API_BACKOFF = 3, 5
API_TIMEOUT_S      = 900
GEN_MAX_OUT, PROG_MAX_OUT, OTHER_MAX_OUT = 12000, 10000, 8000

print(f"examinees        {len(API_EXAMINEES)} API, {len(LOCAL_EXAMINEES)} local")
print(f"judge            {JUDGE}")
print(f"asks for         {K_ASK} terms, {REPS} time(s) each on the API, "
      f"{LOCAL_REPS} on a local examinee")
print(f"writes           {RESULTS_FILE}"
      + ("" if PUSH_RESULTS else "   PUSH DISABLED, copy it out by hand"))


## 1c · Persistent state on Kaggle

In [ ]:
import subprocess

STATE_PATTERNS = ("ctest_*.json", "ctest_*.csv")

STATE_DATASET = ""        # "owner/slug"; empty derives it from KAGGLE_DATASET
STATE_PUSH_EVERY = 3

def _kaggle_ready():
    if ENV != "kaggle":
        return False
    missing = []
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        v = _secret(k)
        if v:
            os.environ[k] = v
        else:
            missing.append(k)
    if missing:
        log(f"  state persistence OFF, missing secret(s): {', '.join(missing)}")
        log("     Add-ons -> Secrets -> add them from your kaggle.json, and turn")
        log("     the toggle on for this notebook. Results still appear in the")
        log("     version output; only state across versions is lost.")
        return False
    return True

KAGGLE_OK = _kaggle_ready()

def _slug():
    if STATE_DATASET:
        return STATE_DATASET
    return f"{os.environ.get('KAGGLE_USERNAME','unknown')}/{KAGGLE_DATASET.lower()}"

def state_files():
    """Every file the push should carry, newest copy of each name.

    Local first, since this run may have just written it. Then whatever is in
    the attached dataset and not local, so another notebook's output survives a
    push from this one."""
    chosen, source = {}, {}
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(pat)):
            chosen[os.path.basename(f)] = f
            source[os.path.basename(f)] = "this run"
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True)):
            b = os.path.basename(f)
            if b not in chosen:
                chosen[b] = f
                source[b] = "carried forward"
    return chosen, source

FRESH_DIR = "/kaggle/working/_fresh"

def _dataset_names():
    """Filenames actually IN the dataset, from the refreshed download.

    NOT from /kaggle/input. The mount is fixed when the session starts, so it
    can be older than the dataset, and comparing the mount against a push
    assembled from the mount compares a thing with itself and can never fire.
    That was the whole failure: the dataset held ctest_metrics.json, the mount
    did not, and a push assembled from the mount deleted it.

    If the refresh did not run or failed, this returns nothing and the guard
    stays quiet, because there is no trustworthy list to compare against.
    """
    out = set()
    if not os.path.isdir(FRESH_DIR):
        return out
    for pat in STATE_PATTERNS:
        for f in glob.glob(os.path.join(FRESH_DIR, pat)):
            out.add(os.path.basename(f))
    return out

STATE_REFRESH = True

def state_refresh():
    """Download the current dataset version into the working directory.

    Cheap, a couple of megabytes, and it removes a whole class of confusion:
    after this the notebook is reading what is IN the dataset rather than what
    happened to be mounted.

    Never raises. A failure leaves /kaggle/input in place, which is the old
    behaviour, and says so.
    """
    if not (STATE_REFRESH and KAGGLE_OK):
        return
    slug = _slug()
    d = FRESH_DIR
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    try:
        r = subprocess.run(["kaggle", "datasets", "download", "-d", slug,
                            "-p", d, "--unzip"],
                           capture_output=True, text=True, timeout=300)
        if r.returncode != 0:
            log(f"  could not refresh from {slug}: "
                f"{(r.stderr or r.stdout).strip()[:120]}")
            log(f"  falling back to whatever the session mounted, which may be "
                f"older than the dataset.")
            return
    except Exception as e:
        log(f"  could not refresh from {slug}: {str(e)[:100]}")
        return

    got, stale = [], []
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(os.path.join(d, pat))):
            b = os.path.basename(f)
            mounted = None
            for p in glob.glob(f"/kaggle/input/**/{b}", recursive=True):
                mounted = p; break
            if os.path.exists(b):
                continue
            shutil.copy(f, b)
            got.append(b)
            if mounted is None:
                stale.append(f"{b} (NOT mounted at all)")
            elif os.path.getsize(mounted) != os.path.getsize(f):
                stale.append(f"{b} (mounted copy is a different size)")
    log(f"  refreshed {len(got)} file(s) from {slug}: {', '.join(sorted(got))}")
    if stale:
        log(f"  the session's mount was OUT OF DATE for: {', '.join(stale)}")
        log(f"  That is why this matters: without the refresh those would have "
            f"been read stale or not at all.")
def state_save(msg="update"):
    """Push a new version of the state dataset. Never raises."""
    if not KAGGLE_OK:
        return
    chosen, source = state_files()
    if not chosen:
        log("  no state files to push yet"); return
    d = "/kaggle/working/_state"
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    for b, f in chosen.items():
        shutil.copy(f, os.path.join(d, b))
    _in_dataset = _dataset_names()
    if not _in_dataset:
        log(f"  note: no refreshed copy of the dataset, so the push cannot be "
            f"checked for deletions. Set STATE_REFRESH = True.")
    _would_lose = _in_dataset - set(chosen)
    if _would_lose:
        log(f"  REFUSING TO PUSH. These are in the dataset and not in this "
            f"push, so the push would DELETE them:")
        for b in sorted(_would_lose):
            log(f"      {b}")
        log(f"  Most likely the session mounted an old version of the dataset. "
            f"STATE_REFRESH pulls the current one at startup; if that failed,")
        log(f"  the log above says why. Nothing has been pushed and nothing is "
            f"lost.")
        return

    carried = [b for b, s in source.items() if s == "carried forward"]
    log(f"  staging {len(chosen)} files"
        + (f", {len(carried)} carried forward from the dataset: "
           f"{', '.join(sorted(carried)[:4])}"
           f"{' ...' if len(carried) > 4 else ''}" if carried else ""))
    slug = _slug()
    json.dump({"title": slug.split("/")[-1], "id": slug,
               "licenses": [{"name": "CC0-1.0"}]},
              open(f"{d}/dataset-metadata.json", "w"))
    try:
        r = subprocess.run(["kaggle", "datasets", "version", "-p", d, "-m", msg,
                            "-r", "zip", "--dir-mode", "zip"],
                           capture_output=True, text=True, timeout=900)
        out = (r.stdout + r.stderr).strip()
        last = out.split("\n")[-1][:160] if out else ""
        if r.returncode == 0 and "error" not in out.lower():
            log(f"  state push: {last}")
        else:
            log(f"  the version push did not go through. Its output was:")
            for _l in out.split("\n")[-6:]:
                if _l.strip(): log(f"    {_l.strip()[:150]}")
            # Only create when the dataset really is absent. "Already in use"
            # means it exists and the version push failed for some other
            # reason, so creating it cannot help.
            missing = ("404" in out or "not found" in out.lower()) and \
                      "already in use" not in out.lower()
            if missing:
                r2 = subprocess.run(["kaggle", "datasets", "create", "-p", d,
                                     "-r", "zip", "--dir-mode", "zip"],
                                    capture_output=True, text=True, timeout=900)
                o2 = (r2.stdout + r2.stderr).strip()
                if "already in use" in o2.lower():
                    log(f"  the dataset exists after all, so the version push "
                        f"failed for another reason. NOTHING WAS PUSHED: "
                        f"download the files from this version's output.")
                else:
                    log(f"  created dataset '{slug}'. Attach it as an input so "
                        f"the next version can read it back.")
                    log(f"  state push: {o2.split(chr(10))[-1][:160]}")
            else:
                log(f"  NOTHING WAS PUSHED. The files are in this version's "
                    f"output and can be downloaded from there.")
    except Exception as e:
        log(f"  state push failed ({str(e)[:80]}). The results are still in this"
            f" version's output.")

stage("PERSISTENT STATE")
if ENV != "kaggle":
    log(f"  not on Kaggle, so nothing to do. Outputs go to {OUTDIR}.")
elif KAGGLE_OK:
    subprocess.run(["pip", "install", "-q", "kaggle"], capture_output=True)
    log(f"  credentials found for '{os.environ['KAGGLE_USERNAME']}'")
    log(f"  state dataset: {_slug()}")
    log(f"  pushes" + (f" every {STATE_PUSH_EVERY} items and" if STATE_PUSH_EVERY
        else "") + " at the end of the run")
_attached = sorted({os.path.basename(os.path.dirname(f))
                    for f in glob.glob("/kaggle/input/*/*")})
if ENV == "kaggle":
    if _attached:
        log(f"  datasets attached: {', '.join(_attached)}")
    else:
        log(f"  NO dataset attached. Add Input -> Datasets -> {KAGGLE_DATASET},")
        log(f"  or the loaders will not find the battery.")
_ch, _src = state_files()
if _ch:
    log(f"  state would carry {len(_ch)} files: {', '.join(sorted(_ch)[:6])}"
        + (" ..." if len(_ch) > 6 else ""))

state_refresh()


## 1d · What the run cost

In [ ]:
import csv, resource

RUNS_CSV = "ctest_runs.csv"

# Seed the local copy from wherever the file already lives, before anything is
# appended. Every other file is read back through find(), which globs
# /kaggle/input; this one was only ever appended, so on Kaggle it started empty
# each session and each run wrote a fresh one-row file. Worse, the dataset push
# prefers the local copy, so that one row REPLACED the accumulated history on
# every push. Rows were being lost, not merely not added.
_prev = find(RUNS_CSV)
if _prev and os.path.abspath(_prev) != os.path.abspath(RUNS_CSV):
    shutil.copy(_prev, RUNS_CSV)
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) carried in from {_prev}")
elif _prev:
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) already here")
else:
    print(f"run history       none yet, {RUNS_CSV} will be created")

_T_START = time.time()
_RUN_KEY = datetime.datetime.now().isoformat(timespec="seconds")
_UNITS_SO_FAR = [0]        # work finished, updated as the run goes
_RUN_NOTE = [""]           # what the row should say
_CPU_START = resource.getrusage(resource.RUSAGE_SELF).ru_utime + \
             resource.getrusage(resource.RUSAGE_SELF).ru_stime
_GPU_SECONDS = [0.0]      # wall clock with a model loaded; add to it when one is

def _gpu_name():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return ""

def account(units=0, note=""):
    """Record this run's cost. Never raises: accounting must not stop work.

    UPSERT, not append. The row is keyed on when the run STARTED, so calling
    this repeatedly updates one row instead of adding several, and the numbers
    only grow. It used to be called once at the end, so a run stopped by hand
    or by the session ending wrote nothing at all and its compute vanished from
    the record.
    """
    try:
        cpu = (resource.getrusage(resource.RUSAGE_SELF).ru_utime +
               resource.getrusage(resource.RUSAGE_SELF).ru_stime) - _CPU_START
        tk = dict(TOKENS) if "TOKENS" in globals() else {}
        row = dict(
            run_started=_RUN_KEY,
            run_ended=datetime.datetime.now().isoformat(timespec="seconds"),
            build=BUILD, env=ENV, note=note,
            wall_s=round(time.time() - _T_START, 1),
            cpu_s=round(cpu, 1),
            gpu_s=round(_GPU_SECONDS[0], 1),
            gpu=_gpu_name(),
            units=units,
            model=(COMPOSER if "COMPOSER" in globals()
                   else REF_MACHINE if "REF_MACHINE" in globals() else ""),
            api_calls=tk.get("calls", 0),
            prompt_tokens=tk.get("in", 0),
            answer_tokens=tk.get("answer", 0),
            thinking_tokens=tk.get("think", 0),
            output_tokens=tk.get("out", 0),
            forward_passes=(REF.calls if "REF" in globals()
                            and hasattr(REF, "calls") else 0),
        )
        old_rows, old_cols = [], []
        if os.path.exists(RUNS_CSV):
            with open(RUNS_CSV, newline="") as f:
                rd = csv.DictReader(f)
                old_cols = list(rd.fieldnames or [])
                old_rows = list(rd)
        old_rows = [r for r in old_rows
                    if r.get("run_started") != _RUN_KEY]
        cols = old_cols + [c for c in row if c not in old_cols]
        with open(RUNS_CSV, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols, restval="")
            w.writeheader()
            for r in old_rows + [row]:
                w.writerow({c: r.get(c, "") for c in cols})
        if OUTDIR != ".":
            shutil.copy(RUNS_CSV, OUTDIR)
    except Exception as e:
        log(f"  accounting failed ({str(e)[:70]}), which changes nothing else")


## 3 · The API

In [ ]:
import logging

from google import genai

CLIENT = genai.Client()

class _AFCCount(logging.Handler):
    def __init__(self):
        super().__init__(); self.n = 0
    def emit(self, record):
        if "automatic function calling" in record.getMessage().lower():
            self.n += 1

_AFC = _AFCCount()

logging.getLogger("google_genai.models").addHandler(_AFC)

TOKENS = {"in": 0, "out": 0, "calls": 0, "empty": 0, "errors": 0, "truncated": 0,
          "think": 0, "answer": 0, "near_ceiling": 0, "tool_tokens": 0}

_TOK_LOCK, _SAVE_LOCK = threading.Lock(), threading.Lock()

def spend_usd():
    with _TOK_LOCK:
        return (TOKENS["in"]*USD_PER_MTOK_IN + TOKENS["out"]*USD_PER_MTOK_OUT)/1e6

def _text_of(r):
    """Thinking models often return the answer in parts rather than in .text."""
    if getattr(r, "text", None): return r.text
    out = []
    for c in (getattr(r, "candidates", None) or []):
        for p in (getattr(getattr(c, "content", None), "parts", None) or []):
            t = getattr(p, "text", None)
            if t and not getattr(p, "thought", False): out.append(t)
    return "".join(out)

def _finish(r):
    for c in (getattr(r, "candidates", None) or []):
        fr = getattr(c, "finish_reason", None)
        if fr is not None: return str(getattr(fr, "name", fr))
    return ""

LAST_CALL = {}

def ask(prompt, temperature=1.0, max_out=None, timeout=None,
        model=None):
    """One metered call. Returns (text, note); note explains an unusual stop.

    `model` defaults to COMPOSER. The evaluator passes a different one per
    examinee, so this cell can stay identical across the notebooks rather than
    each keeping its own copy of the retry and metering logic.
    """
    model = model or COMPOSER
    max_out = max_out or OTHER_MAX_OUT
    if spend_usd() >= MAX_SPEND_USD: return None, "spend cap reached"
    with _TOK_LOCK:
        if TOKENS["in"] + TOKENS["out"] >= TOKEN_BUDGET:
            return None, "token budget reached"
    r = None
    for attempt in range(API_RETRIES):
        try:
            cfg = {"max_output_tokens": max_out, "temperature": temperature,
                   "http_options": {"timeout": int((timeout or API_TIMEOUT_S)*1000)}}
            if DISABLE_AFC:
                cfg["automatic_function_calling"] = {"disable": True}
            r = None
            for drop in ([], ["automatic_function_calling"],
                         ["automatic_function_calling", "http_options"]):
                try:
                    r = CLIENT.models.generate_content(
                        model=model, contents=prompt,
                        config={k: v for k, v in cfg.items() if k not in drop})
                    break
                except TypeError:
                    continue
            if r is None:
                raise TypeError("no accepted config shape")
            break
        except Exception as e:
            m = str(e).lower()
            transient = any(w in m for w in ("unavailable","overloaded","timeout",
                "deadline","timed out","rate limit","resource exhausted",
                "408","429","500","502","503","504"))
            with _TOK_LOCK: TOKENS["errors"] += 1
            if attempt == API_RETRIES-1 or not transient:
                return None, f"api error: {str(e)[:120]}"
            time.sleep(API_BACKOFF * (2**attempt))
    if r is None: return None, "no response"
    u, fr = getattr(r, "usage_metadata", None), _finish(r)
    with _TOK_LOCK:
        TOKENS["calls"] += 1
        if u:
            # Thinking is charged against max_output_tokens and is usually most
            # of it: one 9-digit multiplication cost 1,916 thinking tokens
            # against 18 of answer. That is why a call truncates with nothing
            # usable returned, and it is invisible unless counted apart.
            _think = getattr(u, "thoughts_token_count", 0) or 0
            _ans   = getattr(u, "candidates_token_count", 0) or 0
            TOKENS["in"]  += getattr(u, "prompt_token_count", 0) or 0
            TOKENS["out"] += _ans + _think
            TOKENS["think"] += _think
            TOKENS["answer"] += _ans
            if max_out and (_think + _ans) > 0.85 * max_out:
                TOKENS["near_ceiling"] += 1
            if getattr(u, "tool_use_prompt_token_count", None):
                TOKENS["tool_tokens"] += u.tool_use_prompt_token_count
        if fr and fr not in ("STOP", "FINISH_REASON_STOP"): TOKENS["truncated"] += 1
    # what THIS call cost, for a caller that wants to record it per answer.
    # The totals above are for the run; an examinee's score is not readable
    # without knowing whether its reply was cut off at the cap, and a
    # truncated reply is not empty, it just looks wrong.
    LAST_CALL.update(think=(getattr(u, "thoughts_token_count", 0) or 0) if u else 0,
                     answer=(getattr(u, "candidates_token_count", 0) or 0) if u else 0,
                     prompt=(getattr(u, "prompt_token_count", 0) or 0) if u else 0,
                     finish=fr, max_out=max_out)
    t = _text_of(r)
    if not t:
        with _TOK_LOCK: TOKENS["empty"] += 1
        return None, f"empty reply (finish reason {fr or 'unknown'})"
    return t, ("" if fr in ("STOP","FINISH_REASON_STOP","") else f"finish reason {fr}")

def parse_json(text):
    """Close a truncated object before parsing, since the output budget covers
    the thinking too. Returns (obj, head) - head says why it failed."""
    if not text: return None, ""
    t = re.sub(r"```(?:json)?", "", text)
    i = t.find("{")
    if i < 0: return None, "no opening brace. reply begins: " + t.strip()[:250]
    frag = t[i:]
    for a in (frag, frag+'"]}', frag+"]}", frag+'"}', frag+"}"):
        try: return json.loads(a), ""
        except Exception: pass
    return None, "unparseable json. fragment begins: " + frag[:250]

def tokens_report():
    with _TOK_LOCK: tk = dict(TOKENS)
    extra = "".join([f", {tk['errors']} api errors" if tk["errors"] else "",
                     f", {tk['empty']} empty" if tk["empty"] else "",
                     f", {tk['truncated']} truncated" if tk["truncated"] else "",
                     f", {tk['near_ceiling']} near the output ceiling"
                     if tk["near_ceiling"] else ""])
    share = tk["think"] / max(1, tk["think"] + tk["answer"])
    log(f"  composer: {tk['calls']} calls, {tk['in']+tk['out']:,} tokens "
        f"(~${spend_usd():.2f}){extra}")
    if tk["think"]:
        log(f"            thinking {tk['think']:,} of {tk['think']+tk['answer']:,} "
            f"output tokens ({share:.0%}); the model has no interpreter, so a long "
            f"sequence is simulated step by step")
    if tk["tool_tokens"]:
        log(f"            {tk['tool_tokens']:,} TOOL tokens: something IS calling "
            f"a tool, which changes what these runs mean")

def parallel(fn, jobs, workers=None, label=""):
    """Run fn over jobs. Results follow jobs, not completion order."""
    workers = workers or MAX_WORKERS
    if workers <= 1 or len(jobs) <= 1:
        return [fn(j) for j in jobs]
    out = [None] * len(jobs)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(fn, j): i for i, j in enumerate(jobs)}
        done = 0
        for f in as_completed(futs):
            i = futs[f]
            try: out[i] = f.result()
            except Exception as e:
                out[i] = None
                log(f"      worker failed on job {i}: {str(e)[:110]}")
            done += 1
            if label and done % max(1, len(jobs)//10) == 0:
                log(f"      {label}: {done}/{len(jobs)}")
    return out

stage("LOADING")

_renamed = 0

if _renamed:
    log(f"  renamed `rule` to `seed` on {_renamed} items from an earlier battery")

EXCLUSIONS = ("duplicate", "quota", "flawed", "stale", "questionable")

_STOP = {"the", "each", "term", "terms", "subsequent", "starting", "start",
         "with", "next", "and", "for", "from", "list", "listed", "order",
         "their", "that", "this", "into", "then", "given", "using", "based"}

def _overlap(a, b):
    return len(a & b) / len(a | b) if (a and b) else 0.0

# ---------------------------------------------------------------------------
#  Three providers behind one call
#
#  ask() speaks to Gemini through google.genai. An examinee from another house
#  needs a different client and returns a different shape, including where the
#  reasoning tokens are reported, which every row now records. Rather than
#  three copies of the retry and metering logic, the provider is chosen from
#  the model name and each adapter returns the same tuple.
#
#  The judge stays on Gemini whatever the examinees are, so a score means the
#  same thing across providers and across the runs already made.

OPENAI_MODELS    = ("gpt-", "o1", "o3", "o4")
ANTHROPIC_MODELS = ("claude-",)

_CLIENTS = {}

# AN EXAMINEE IS A MODEL AND ITS SETTINGS, and a setting that changes the
# answers makes a different examinee. Naming one "claude-opus-5-fallback" runs
# the same model with server-side fallbacks asked for, and its rows sit beside
# the plain ones under their own name instead of overwriting them. So both can
# be scored, compared and reported, and the gap check treats them as two
# examinees because that is what they are.
FALLBACK_SUFFIX = "-fallback"

def _wants_fallback(name):
    return str(name or "").endswith(FALLBACK_SUFFIX)

def base_model(name):
    """The name the API knows. The suffix is ours, not the provider's."""
    n = str(name or "")
    return n[:-len(FALLBACK_SUFFIX)] if _wants_fallback(n) else n

def provider_of(model):
    m = base_model(model).lower()
    if m.startswith(ANTHROPIC_MODELS): return "anthropic"
    if any(m.startswith(p) for p in OPENAI_MODELS): return "openai"
    return "google"

# The environment variable each house reads. Change it HERE if a secret is
# named differently, and nowhere else: the preamble copies whatever is listed
# here out of the Kaggle secrets, so the two cannot drift apart.
KEY_FOR = {"openai": "OPENAI_API_KEY", "anthropic": "CLAUDE_API_KEY"}

def _client(kind):
    if kind in _CLIENTS: return _CLIENTS[kind]
    var = KEY_FOR[kind]
    key = os.environ.get(var)
    if not key:
        raise RuntimeError(
            f"{var} is not set, so {kind} cannot be reached. A Kaggle secret "
            f"is not an environment variable until something fetches it, and "
            f"the preamble fetches exactly the names in KEY_FOR. Check that "
            f"the secret is called {var} in Settings, that it is toggled on "
            f"for this notebook, and that the preamble printed "
            f"'{var} found'. If your secret has another name, change KEY_FOR "
            f"and nothing else.")
    if kind == "openai":
        from openai import OpenAI
        _CLIENTS[kind] = OpenAI(api_key=key)
    elif kind == "anthropic":
        import anthropic
        _CLIENTS[kind] = anthropic.Anthropic(api_key=key)
    return _CLIENTS[kind]

def ask_any(prompt, model, temperature=0.0, max_out=2000, timeout=None):
    """One call to whichever house owns `model`.

    Returns (text, note, usage) with usage in the same shape ask() records:
    think, answer, prompt, finish, max_out. A provider that does not report
    reasoning tokens separately reports think = 0, which is true of it rather
    than a guess.
    """
    kind = provider_of(model)
    _fb = _wants_fallback(model)      # asked for by the examinee's NAME
    model = base_model(model)         # what the provider is actually sent
    # Set for EVERY provider, not only the one that can use them. Both are
    # recorded on every row, and the Anthropic branch is the only place they
    # can become true, so an OpenAI or Google row reading False is the honest
    # answer rather than a missing name.
    _fb_used = False
    if kind == "google":
        txt, note = ask(prompt, temperature=temperature, max_out=max_out,
                        timeout=timeout, model=model)
        return txt, note, dict(LAST_CALL)

    t0 = time.time()
    for attempt in range(API_RETRIES):
        try:
            if kind == "openai":
                r = _client("openai").chat.completions.create(
                    model=model, messages=[{"role": "user", "content": prompt}],
                    max_completion_tokens=max_out,
                    timeout=timeout or API_TIMEOUT_S)
                txt = (r.choices[0].message.content or "")
                fin = r.choices[0].finish_reason
                u = r.usage
                # the reasoning models report their thinking here; the others
                # leave it absent, which reads as 0
                think = 0
                d = getattr(u, "completion_tokens_details", None)
                if d is not None:
                    think = getattr(d, "reasoning_tokens", 0) or 0
                # what was ASKED for and what was USED are different things:
                # an SDK that cannot pass the argument still answers, and those
                # rows must not be counted as a fallback run
                use = dict(fallbacks=bool(_fb_used), fallbacks_asked=bool(_fb),
                           think=int(think),
                           answer=int((u.completion_tokens or 0) - think),
                           prompt=int(u.prompt_tokens or 0),
                           finish=("MAX_TOKENS" if fin == "length" else
                                   str(fin).upper()),
                           max_out=max_out)
            else:
                # temperature is dropped if the SDK or the model refuses it.
                # Some builds of the Anthropic client reject the argument
                # outright, and a run must not fail over a decoding setting it
                # cannot apply. Whether it was applied is recorded, because a
                # score from three attempts means something different when the
                # sampling could not be pinned.
                _kw = dict(model=model, max_tokens=max_out,
                           messages=[{"role": "user", "content": prompt}])
                _temp_ok = True

                def _create(**extra):
                    kw = dict(_kw, **extra)
                    try:
                        return _client("anthropic").messages.create(
                            temperature=temperature, **kw), True
                    except TypeError as te:
                        if "temperature" not in str(te): raise
                        return _client("anthropic").messages.create(**kw), False

                if _fb:
                    for _route, _extra in (
                        # 1. the beta namespace, which is where a beta belongs
                        ("beta namespace", None),
                        # 2. the same arguments on the ordinary endpoint, in
                        #    case this SDK accepts them there
                        ("messages.create", dict(
                            fallbacks="default",
                            betas=["server-side-fallback-2026-07-01"])),
                        # 3. raw, for an SDK that knows neither but will pass
                        #    through what it is given
                        ("extra_body", dict(
                            extra_body={"fallbacks": "default"},
                            extra_headers={"anthropic-beta":
                                           "server-side-fallback-2026-07-01"})),
                    ):
                        try:
                            if _extra is None:
                                _b = _client("anthropic").beta.messages.create
                                try:
                                    r = _b(temperature=temperature,
                                           fallbacks="default",
                                           betas=["server-side-fallback-2026-07-01"],
                                           **_kw)
                                except TypeError as te:
                                    if "temperature" not in str(te): raise
                                    _temp_ok = False
                                    r = _b(fallbacks="default",
                                           betas=["server-side-fallback-2026-07-01"],
                                           **_kw)
                            else:
                                r, _temp_ok = _create(**_extra)
                            _fb_used = True
                            break
                        except (TypeError, AttributeError) as te:
                            _last = f"{_route}: {str(te)[:70]}"
                            continue
                    if not _fb_used:
                        log(f"    fallbacks not available in this SDK "
                            f"({_last}); answering without them, so these rows "
                            f"are NOT a fallback run")
                        r, _temp_ok = _create()
                else:
                    r, _temp_ok = _create()
                txt = "".join(b.text for b in r.content
                              if getattr(b, "type", "") == "text")
                think = sum(len(getattr(b, "thinking", "")) // 4
                            for b in r.content
                            if getattr(b, "type", "") == "thinking")
                # what was ASKED for and what was USED are different things:
                # an SDK that cannot pass the argument still answers, and those
                # rows must not be counted as a fallback run
                use = dict(fallbacks=bool(_fb_used), fallbacks_asked=bool(_fb),
                           think=int(think),
                           answer=int((r.usage.output_tokens or 0) - think),
                           prompt=int(r.usage.input_tokens or 0),
                           finish=("MAX_TOKENS" if r.stop_reason == "max_tokens"
                                   else str(r.stop_reason).upper()),
                           temp_applied=_temp_ok,
                           max_out=max_out)
            with _TOK_LOCK:
                TOKENS["in"] += use["prompt"]
                TOKENS["out"] += use["answer"] + use["think"]
                TOKENS["calls"] += 1
            return txt, None, use
        except Exception as e:
            msg = str(e)[:140]
            if attempt == API_RETRIES - 1:
                return None, f"{kind} error: {msg}", {}
            time.sleep(API_BACKOFF * (attempt + 1))
    return None, "unreachable", {}


## 4 · Open-weight examinees

Only used when `LOCAL_EXAMINEES` is not empty. The class is the reference machine's, without the pricing: an examinee only has to answer.

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList
# at the top of the cell: _StopLocal subclasses StoppingCriteria at
# class-definition time, so the import cannot sit inside a method.

# ---------------------------------------------------------------------------
# WHEN TO STOP A LOCAL GENERATION
#
# The annotator stops as soon as the required terms appear, and its docstring
# names the case this notebook ran into: a machine that never closes its
# reasoning has nothing after a marker to read, so the terms in its working
# are the only answer there is, and refusing to stop on them costs the full
# budget every time. Phi-4-reasoning reaches the answer within a few hundred
# tokens and then repeats "I'll produce answer: ..." until the budget runs
# out, 8000 tokens and forty minutes per item. With no criterion at all this
# notebook paid that on every reply.
#
# Two conditions, and both are recorded so a row says which fired.
#
#   TERMS   the k requested terms are present, in order. This is the
#           annotator's condition. It is not neutral for an examinee: a
#           correct run is cut short and an incorrect one is not. It is used
#           here because the alternative is not measuring this model at all,
#           and because the terms being present is what the score asks.
#
#   LOOP    the last 60 characters have already appeared three times. This
#           looks at repetition and not at correctness, so it treats a right
#           and a wrong answer alike.
#
# A model that answers and stops, which is all twelve API examinees and
# Qwen3-14B, never reaches either condition and is unaffected.
LOCAL_STOP_TERMS = False  # the early-stopping rule was for Phi-4 as an examinee (Appendix E);
                          # the examinees of the paper ran without it
LOCAL_STOP_LOOP  = False
LOCAL_STOP_EVERY = 32     # tokens between checks; decoding is not free
LOCAL_STOP_MIN   = 64     # never stop before this, so a model restating the
                          # instruction is not cut off mid-sentence
LOCAL_STOP_LOOP_AFTER = 2000   # tokens before the loop condition may fire

def _norm_term(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def _terms_in_order(txt, truth):
    """How many of `truth` appear as consecutive comma-separated FIELDS of
    some line of `txt`, in order.

    The first version stripped everything but letters and digits and searched
    the whole reply, which let short terms match by accident: for truth
    `s, cl, ar, k, ca` those letters occur in order in almost any English
    sentence, so the criterion stopped Phi-4-reasoning in the middle of its
    reasoning, before it had written a single term. A term now has to be a
    whole field, compared after normalising each field alone.
    """
    want = [_norm_term(t) for t in truth]
    if not want: return 0
    last = len(want) - 1
    def _fits(field, n, first):
        # The first field of a run may carry what came before it on the line
        # ("ANSWER: s" becomes "answers"), and the field holding the last
        # term may carry what follows ('10" on final line'). Every field in
        # between has to match exactly, which is what keeps short terms from
        # matching by accident.
        if field == want[n]: return True
        if first and field.endswith(want[n]): return True
        if n == last and field.startswith(want[n]): return True
        return False
    best = 0
    for line in str(txt).splitlines():
        fields = [_norm_term(f) for f in re.split(r"[,;]", line)]
        fields = [f for f in fields if f]
        for s in range(len(fields)):
            n = 0
            while (n < len(want) and s + n < len(fields)
                   and _fits(fields[s + n], n, n == 0)):
                n += 1
            best = max(best, n)
            if best == len(want): return best
    return best

class Examinee:
    """An open-weight examinee. Generates; it is never asked to price."""

    THINK_END = {"qwen": "</think>", "olmo": "</think>",
                 "phi-4-reasoning": "</think>", "phi-4": None,
                 "deepseek": "</think>", "gemma": None, "falcon": None}

    def __init__(self, name):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.name, self.torch = name, torch
        self.tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        # THE SAME LOADING THE REFERENCE MACHINES GET, when asked for. A
        # reference machine runs 4-bit nf4 with double quantisation and
        # bfloat16 compute; an examinee sitting the battery at float16 would
        # differ from it in the setup as well as in the task, and the point of
        # running a reference machine as an examinee is to see whether the
        # measure favours models like itself. Any difference should be the
        # task.
        # WORD FOR WORD WHAT THE REFERENCE MACHINES GET. The Machine class in
        # the unquestionability and annotation notebooks loads with
        # trust_remote_code, device_map auto, and either 4-bit nf4 with double
        # quantisation and bfloat16 compute, or bfloat16. Not float16, which is
        # a third thing neither of them uses.
        kw = dict(trust_remote_code=True, device_map="auto")
        if LOCAL_4BIT:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
        else:
            kw["torch_dtype"] = torch.bfloat16
        # NO FALLBACK. Loading a reference machine in a precision the
        # reference machine does not use defeats the run: the question is
        # whether the measure favours models like the one that made it, and an
        # answer from different weights is not an answer. It also silently
        # wrecks the timing, since an 8B in float16 does not fit this card and
        # device_map spills it to the CPU at about a token a second.
        self.model = AutoModelForCausalLM.from_pretrained(name, **kw)
        self.model.eval()
        self.LAST = {}
        self.think_end = next((v for k, v in self.THINK_END.items()
                               if k in name.lower()), None)

        # WHERE A TURN ENDS. Some chat templates close a turn with a token the
        # tokenizer does not report as eos, and generate() then has no reason
        # to stop: Phi-4-reasoning ran to the full 8000 tokens on every reply,
        # 40 minutes each, where Qwen3-14B had a median of 752. Any candidate
        # the tokenizer knows is added to the eos set.
        _cands = ["<|im_end|>", "<|end|>", "<|eot_id|>",
                  "<|end_of_text|>", "<|endoftext|>"]
        _ids = [self.tok.eos_token_id] + [self.tok.convert_tokens_to_ids(t)
                                          for t in _cands]
        self.eos_ids = sorted({i for i in _ids
                               if isinstance(i, int) and i >= 0
                               and i != self.tok.unk_token_id})
        log(f"    ends a turn on {self.eos_ids} = "
            f"{[self.tok.decode([i]) for i in self.eos_ids]}")
        log(f"    loaded {name}"
            + (f", will look for {self.think_end!r}; a model that does not "
               f"emit it reports think = 0" if self.think_end
               else ", no reasoning marker, so think is reported as 0"))

    class _StopLocal(StoppingCriteria):
        """Stop on the terms, or on a repeating tail. Records which."""
        def __init__(self, tok, n0, truth):
            self.tok, self.n0, self.truth = tok, n0, list(truth or [])
            self.i = 0
            self.why = None

        def __call__(self, input_ids, scores, **kw):
            self.i += 1
            if self.i < LOCAL_STOP_MIN or self.i % LOCAL_STOP_EVERY:
                return False
            txt = self.tok.decode(input_ids[0][self.n0:],
                                  skip_special_tokens=True)
            if LOCAL_STOP_TERMS and self.truth and \
                    _terms_in_order(txt, self.truth) >= len(self.truth):
                self.why = "terms"; return True
            if LOCAL_STOP_LOOP and self.i >= LOCAL_STOP_LOOP_AFTER \
                    and len(txt) > 1000:
                tail = txt[-200:]
                if txt.count(tail) >= 5:
                    self.why = "loop"; return True
            return False

    def generate(self, prompt, max_new_tokens=None, truth=None):
        """Generate, and record what it cost, so a local examinee reports the
        same budget fields an API one does."""
        max_new_tokens = max_new_tokens or 512
        msgs = [{"role": "user", "content": prompt}]
        try:
            # thinking on where the template offers it, since the reference
            # machines run with it and the tag says so: qwen3-14b-4bit-THINK
            try:
                text = self.tok.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=True,
                    enable_thinking=LOCAL_THINK)
            except TypeError:
                text = self.tok.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            text = prompt
        enc = self.tok(text, return_tensors="pt").to(self.model.device)
        with self.torch.no_grad():
            _sc = self._StopLocal(self.tok, enc["input_ids"].shape[1], truth)
        out = self.model.generate(**enc, max_new_tokens=max_new_tokens,
                                      do_sample=False,
                                      eos_token_id=(self.eos_ids or
                                                    self.tok.eos_token_id),
                                      pad_token_id=self.tok.pad_token_id,
                                      stopping_criteria=StoppingCriteriaList([_sc]))
        new = out[0][enc["input_ids"].shape[1]:]
        n_new = int(new.shape[0])
        full = self.tok.decode(new, skip_special_tokens=True)

        # Split reasoning from answer on the model's own marker, the same way
        # the annotator does for the reference machine. Without this a local
        # examinee reports all its tokens as answer, and the reasoning length
        # is exactly what a compute-against-accuracy reading needs. A model
        # with no marker reports think = 0, which is true of it.
        think_txt = ""
        if self.think_end:
            j = full.rfind(self.think_end)
            if j >= 0:
                think_txt, full = full[:j], full[j + len(self.think_end):]
        n_think = (len(self.tok.encode(think_txt, add_special_tokens=False))
                   if think_txt else 0)
        self.LAST = dict(stopped_by=_sc.why, think=n_think,
                         answer=n_new - n_think,
                         prompt=int(enc["input_ids"].shape[1]),
                         finish="LENGTH" if n_new >= max_new_tokens else "STOP",
                         max_out=max_new_tokens,
                         think_share=round(n_think / n_new, 3) if n_new else None)
        return full

    def unload(self):
        del self.model
        import gc; gc.collect()
        try: self.torch.cuda.empty_cache()
        except Exception: pass
        log(f"    unloaded {self.name}")


## 5 · Asking and scoring

In [ ]:
EXAM_PROMPT = ("Continue the sequence with the next {k} terms only.\n"
               "Sequence: {seq}\n"
               "Think if you need to, but end your reply with the answer "
               "alone on the final line, the {k} terms separated by commas "
               "and nothing else.\nNext {k} terms:")

JUDGE_PROMPT = ("A sequence puzzle was shown with this prefix:\n  {seq}\n"
                "The intended next terms are:\n  {truth}\n"
                "An examinee answered:\n  {resp}\n\n"
                "How many of the intended terms did the examinee give, in "
                "order, starting from the first? Wording and spacing do not "
                "matter; the term does. Stop counting at the first one that "
                "is wrong or missing.\n"
                "Reply with a single number and nothing else.")

# Four cases with known answers, run before anything is spent. A judge that
# fails these is not scoring, and the run stops rather than producing numbers
# that look like results.
JUDGE_PROBES = [("a, b, c", "d, e, f", "d, e, f", 3),
                ("a, b, c", "d, e, f", "The answer is: d, e, f", 3),
                ("a, b, c", "d, e, f", "d, e, x", 2),
                ("a, b, c", "d, e, f", "x, y, z", 0)]

def pairs(spec):
    """(name, budget) from either form, so a bare name still works."""
    out = []
    for e in spec:
        if isinstance(e, (tuple, list)):
            out.append((e[0], int(e[1])))
        else:
            out.append((e, DEFAULT_MAX_OUT))
    return out

def looks_looped(text, tail=400, unit=60):
    """Cheap check for a model repeating itself at the end of its output.

    Being CUT is measured: the model was stopped before it emitted its end
    token, and the finish reason says so. LOOPING is an inference, because a
    model cut off may have been repeating itself or may have been working
    steadily and simply run out of room, and those are different failures.

    So this is reported separately and never as the reason for the cut. It
    takes the last `tail` characters and asks whether a chunk of `unit` of
    them appears earlier in that tail.
    """
    t = " ".join((text or "").split())[-tail:]
    if len(t) < 2 * unit:
        return False
    return t[-unit:] in t[:-unit]

def last_line(raw):
    """The answer, from a reply that may think first.

    Takes the last non-empty line, but never lets tidying empty a real answer:
    it falls back through the last line, the whole text, then the raw string.
    """
    raw = (raw or "").strip()
    for line in reversed(raw.split("\n")):
        line = re.sub(r"[*`_]", "", line).strip(" -:")
        if line:
            return line
    return re.sub(r"[*`_]", "", raw).strip() or raw

def ask_examinee(model, item, budget):
    """One attempt. Returns the answer and the raw reply."""
    txt, note, use = ask_any(EXAM_PROMPT.format(
        seq=", ".join(item["x_pref"]), k=len(item["truth"])),
        model=model, temperature=EXAM_TEMP, max_out=budget)
    if txt is None:
        return "", f"(no reply: {note})", use
    return last_line(txt)[:300], txt[:600], use

def score(seq, truth, resp):
    """How many terms the examinee got, in order, before the first mistake.

    Judged rather than string-matched, because "10 metres" and "10m" are the
    same term and an examinee should not be marked down for spacing. An
    unreadable verdict counts as 0, which can only make an examinee look worse
    and never better.
    """
    txt, note = ask(JUDGE_PROMPT.format(seq=", ".join(seq[-8:]),
                                        truth=", ".join(truth),
                                        resp=resp or "(no answer)"),
                    model=JUDGE, temperature=0.0, max_out=2000)
    m = re.search(r"\d+", (txt or ""))
    return min(int(m.group(0)), len(truth)) if m else 0


## 6 · Ordering and administering

In [ ]:
def spread_order(items):
    """Farthest-point order over (r_hat, t_hat, log K), each axis scaled.

    So that ANY PREFIX is a sample. A run stopped after an hour still covers
    the grid and the cost range, and a correlation computed on what it managed
    is not the correlation of one corner. Same ordering the annotator uses, so
    the two runs walk the battery the same way.
    """
    P = {}
    for it in items:
        k = it.get("K") or 0
        P[it["id"]] = (it.get("r_hat"), it.get("t_hat"),
                       math.log(k) if k > 0 else None)
    good = [i for i in items if None not in P[i["id"]]]
    rest = [i for i in items if None in P[i["id"]]]
    if len(good) < 2:
        return items
    lo = [min(P[i["id"]][d] for i in good) for d in range(3)]
    hi = [max(P[i["id"]][d] for i in good) for d in range(3)]
    at = lambda i: tuple((P[i["id"]][d] - lo[d]) / (hi[d] - lo[d])
                         if hi[d] > lo[d] else 0.0 for d in range(3))
    out = [max(good, key=lambda i: sum(at(i)))]
    pool = [i for i in good if i is not out[0]]
    while pool:
        nxt = max(pool, key=lambda i: min(
            sum((a - b) ** 2 for a, b in zip(at(i), at(j))) for j in out))
        out.append(nxt); pool.remove(nxt)
    return out + rest

def administer(examinee, budget, is_local=False, model=None, items=None):
    """Every item, REPS times, scored. Returns the rows.

    `items` defaults to all of them. Passing a subset is how an examinee is
    asked only for what it has not answered yet.
    """
    rows = []
    _saved = [0]   # how many of `rows` are already in RESULTS
    _its = ITEMS if items is None else items
    # one attempt on the local path: see LOCAL_REPS
    _reps = LOCAL_REPS if is_local else REPS
    jobs = [(it, rep) for it in _its for rep in range(_reps)]

    # A local examinee generates one at a time with nothing between, so an
    # hour of silence says neither that it is working nor that it is not. This
    # reports every tenth, with the tokens the last one spent, which is what
    # distinguishes a model answering in 1500 tokens from one running to the
    # cap every time.
    # ITS OWN NAME. The progress loop below binds _n as the job index, in this
    # same scope, so by the time one() ran the name held an int and the first
    # call raised on _n[0]. The local examinees never got past their first
    # generation.
    _calls = [0]

    def one(job):
        it, rep = job
        if is_local:
            # the truth goes in so the criterion can stop once the terms
            # are there; see LOCAL_STOP_TERMS for why that is not neutral
            full = model.generate(EXAM_PROMPT.format(
                seq=", ".join(it["x_pref"]), k=len(it["truth"])),
                max_new_tokens=budget, truth=it["truth"])
            _calls[0] += 1
            if _calls[0] % 10 == 0 or _calls[0] == 1:
                _u = model.LAST or {}
                log(f"      {examinee}: {_calls[0]}/{len(jobs)}, last reply "
                    f"{(_u.get('think', 0) or 0) + (_u.get('answer', 0) or 0)}"
                    f" tokens of {budget}")
            use = dict(model.LAST or {})
            # WHAT THE JUDGE SEES. An examinee that answers and stops puts
            # the terms on its last line, as the prompt asks, and the last
            # line is what every examinee is marked on. One that the
            # stopping criterion cut off has no last line to speak of: the
            # criterion fires when the terms appear, wherever they appear,
            # so they may be mid-reply. The judge then sees the whole reply,
            # as the annotator's VERIFY_PROMPT does when it accepts terms
            # "as its answer or in its working". Marking only the last line
            # after a cut scored Phi-4-reasoning zero on items it continues
            # unaided as a reference machine.
            if use.get("stopped_by"):
                ans = full[-3000:]
            else:
                ans = last_line(full)[:300]
            raw = full[:600]
            _tail = full
        else:
            ans, raw, use = ask_examinee(examinee, it, budget)
            _tail = raw
        got = score(it["x_pref"], it["truth"], ans)
        # tokens SPENT and whether the cap was hit, per answer. A reply cut off
        # at the budget scores as wrong and is not wrong; unlike a timeout it
        # leaves text behind, so nothing announces it.
        _out = (use.get("think", 0) or 0) + (use.get("answer", 0) or 0)
        # CUT is measured: the model was stopped before its end token, and the
        # finish reason says so. LOOPED is a guess from the tail, reported
        # beside it and never in place of it, because a cut model may have been
        # repeating itself or may have been working steadily and run out of
        # room, and those are different failures.
        _fin = str(use.get("finish") or "")
        _cut = _fin.upper() in ("LENGTH", "MAX_TOKENS", "FINISH_REASON_MAX_TOKENS")
        return dict(item=it["id"], examinee=examinee, budget=budget, rep=rep,
                    n_correct=got, k=len(it["truth"]),
                    answer=ans, raw=raw,
                    tok_out=_out, tok_think=use.get("think"),
                    think_share=(round((use.get("think") or 0) / _out, 3)
                                 if _out else None),
                    tok_prompt=use.get("prompt"),
                    finish=_fin or None,
                    temp_applied=use.get("temp_applied", True),
                    was_cut=_cut,
                    stopped_by=use.get("stopped_by"),
                    near_budget=bool(budget and _out > 0.95 * budget),
                    looked_looped=looks_looped(_tail) if _cut else None,
                    max_out=budget,
                    # ASKED and USED are separate. An SDK that cannot pass the
                    # argument still answers, and those rows are not a fallback
                    # run whatever the examinee is called. The row is built
                    # from named keys, so a key added to `use` and not named
                    # here is silently dropped, which is what happened the
                    # first time these two were added.
                    fallbacks_asked=use.get("fallbacks_asked", False),
                    fallbacks=use.get("fallbacks", False),
                    cell=list(it["cell"]), r_hat=it["r_hat"],
                    t_hat=it["t_hat"], K=it["K"], h=it["h"], n_M=it["n_M"])

    if is_local:
        # A counter. The API path gets one from parallel(); this loop is
        # serial and printed nothing, so a local examinee ran for half an hour
        # with no way to tell how much was left or whether it had stuck.
        _t0 = time.time()
        for _n, j in enumerate(jobs, 1):
            rows.append(one(j))
            if _n % 5 == 0 or _n == len(jobs):
                _el = time.time() - _t0
                _cut = sum(1 for r in rows if r.get("was_cut"))
                log(f"      {_n}/{len(jobs)}   {_el/60:.1f} min, "
                    f"~{_el/_n*(len(jobs)-_n)/60:.0f} left"
                    + (f", {_cut} cut at the budget" if _cut else ""))

            if STATE_PUSH_EVERY and _n % STATE_PUSH_EVERY == 0 \
                    and _n < len(jobs):
                RESULTS.extend(rows[_saved[0]:_n])
                _saved[0] = _n
                save_results()
                if PUSH_MIDRUN and PUSH_RESULTS:
                    state_save(f"{BUILD}: {len(RESULTS)} rows, mid-run")
                    log(f"      saved and pushed at {_n}/{len(jobs)}")
                else:
                    log(f"      saved at {_n}/{len(jobs)}")

        # whatever the last block did not cover
        RESULTS.extend(rows[_saved[0]:])
        _saved[0] = len(rows)
    else:
        rows = [r for r in parallel(one, jobs, workers=MAX_WORKERS,
                                    label=examinee.split("/")[-1]) if r]
    return rows


## 7 · Writing the results

In [ ]:
RESULTS = []   # filled by the run; declared here so save_results can be
               # called after every examinee rather than only at the end

def _merged_rows():
    """This run's rows, on top of whatever was recorded before.

    FILL_GAPS_ONLY asks each examinee only for what it is missing, so a run
    produces a handful of rows and not a battery's worth. Writing those alone
    replaces the file: one run wrote twelve rows over two thousand two hundred,
    and pushing it would have destroyed every earlier measurement.

    Keyed on examinee, item, repetition and the prefix it was asked at, since
    an item asked again at a longer prefix is a different measurement and both
    are kept.
    """
    try:
        prev = json.load(open(find(RESULTS_FILE) or RESULTS_FILE))["rows"]
    except Exception:
        prev = []
    key = lambda r: (r["examinee"], r["item"], r["rep"], r.get("n_M"))
    fresh = {key(r) for r in RESULTS}
    return [r for r in prev if key(r) not in fresh] + list(RESULTS)

def save_results():
    """Write the rows. Called after every examinee, not only at the end.

    An examinee is 30 minutes of API calls and there is no reason to risk it
    on the run finishing.
    """
    doc = dict(_README=(
        "One row per item, examinee and repetition. n_correct is how many of "
        "the k asked terms the examinee gave in order before its first "
        "mistake, judged rather than string-matched, so wording and spacing "
        "do not count against it. The prefix shown is the item's n_M terms, "
        "the number at which it certified. Every examinee gets the same "
        "budget, exam_max_out output tokens with thinking included, and each "
        "row records what it spent and whether it hit the cap: a reply cut "
        "off at the budget scores as wrong without being wrong."),
        battery=BATTERY, machine=MACHINE_ID, judge=JUDGE,
        k_ask=K_ASK, reps=REPS, exam_temp=EXAM_TEMP,
        # per examinee, since an examinee is a model AND a budget
        budgets={n: b for n, b in pairs(API_EXAMINEES) + pairs(LOCAL_EXAMINEES)},
        written_by=BUILD,
        written_on=datetime.datetime.now().isoformat(timespec="minutes"),
        rows=_merged_rows())
    with open(RESULTS_FILE, "w") as f:
        json.dump(doc, f, indent=1)


## 8 · Setting up, and is the judge scoring?

In [ ]:
stage("WHAT THIS RUN USES")
_local = bool(LOCAL_EXAMINEES)
log(f"  API examinees   "
    + (", ".join(f"{n} at {b}" for n, b in pairs(API_EXAMINEES)) or "none"))
log(f"  local examinees "
    + (", ".join(f"{n} at {b}" for n, b in pairs(LOCAL_EXAMINEES)) or "none"))
log(f"  an examinee is a model AND a budget: the same model at two budgets is "
    f"two examinees, and the pair is what is recorded.")
if _local:
    log(f"\n  THIS RUN NEEDS A GPU. Local examinees are loaded one at a time, "
        f"and each is a download of several GB.")
    log(f"  An API-only run needs none, so if the GPU is wanted elsewhere, "
        f"empty LOCAL_EXAMINEES and this becomes a CPU run.")
else:
    log(f"\n  THIS RUN NEEDS NO GPU. Everything goes through the API, so it "
        f"can run beside a job that does need one.")
    log(f"  Add to LOCAL_EXAMINEES to include open-weight examinees, and the "
        f"run will want a GPU.")

if JUDGE in API_EXAMINEES or JUDGE in LOCAL_EXAMINEES:
    log(f"\n  NOTE: the judge, {JUDGE}, is also an examinee, so it marks its "
        f"own answers. It is asked how many terms match a truth it is given,")
    log(f"  not whether the answer is good, which is a narrow enough question "
        f"that this is survivable. But if one examinee scores oddly well, "
        f"this is")
    log(f"  the first thing to rule out: put a model outside the examinee "
        f"list in JUDGE and compare.")

if not PUSH_RESULTS:
    log(f"\n  ###############################################################")
    log(f"  #  PUSH DISABLED. Results are written to {RESULTS_FILE}")
    log(f"  #  in this session ONLY and are NOT pushed to the dataset.")
    log(f"  #  COPY THEM OUT BEFORE THE SESSION ENDS or they are lost.")
    log(f"  #  A push stages every file, including ones this run only read,")
    log(f"  #  so pushing while an annotation run is going would undo it.")
    log(f"  ###############################################################")

# ---------------------------------------------------------------------------
stage("THE BATTERY")
BAT = json.load(open(find(BATTERY) or BATTERY))
_items = BAT["items"]
try:
    MET = json.load(open(find(METRICS) or METRICS))
    MET = MET.get("items", MET)
except Exception:
    MET = {}
log(f"  {len(_items)} items in the battery, {len(MET)} with metrics")

def _is_valid(b):
    return (b.get("valid") is True
            or (b.get("repro", {}).get("seed_cold") or {}).get("verdict")
            == "reproduced")

_valid = {b["id"] for b in _items if _is_valid(b)}
log(f"  {len(_valid)} of {len(_items)} items are valid")

_tags = sorted({k[4:] for b in _items for k in b if k.startswith("n_M_")})
log(f"  machines that have priced this battery: {', '.join(_tags) or 'none'}")

def _certified_anywhere(b):
    return any(b.get(f"n_M_{t}") is not None for t in _tags)

def _prefix_for(b):
    """The prefix to administer: the LONGEST any machine needed.

    An item is unquestionable at a given length under a given machine. Two
    machines can want different lengths, and a prefix long enough for the
    strictest is long enough for all of them, so the answers stay usable
    whichever machine the analysis is relative to.

    The shorter prefix would make the item questionable under the stricter
    machine: a rival reproduces it and then diverges, so an examinee answering
    differently may not be wrong. It is also the easier task, which is why this
    is not free: a longer prefix shows more of the pattern.
    """
    lens = [b[f"n_M_{t}"] for t in _tags if b.get(f"n_M_{t}") is not None]
    return max(lens) if lens else None

ITEMS = []
for b in _items:
    # ITEM_ONLY names the items to administer and overrides every other
    # filter below, for re-running one row rather than a battery.
    if ITEM_ONLY and b["id"] not in ITEM_ONLY:
        continue
    if VALID_ONLY and b["id"] not in _valid:
        continue
    nM = b.get(f"n_M_{MACHINE_TAG}")
    if ONLY_CERTIFIED and nM is None:
        # the prefix has to come from somewhere: use whichever machine
        # certified it, since that is the length at which it is unquestionable
        if not (CERTIFIED_ANY_MACHINE and _certified_anywhere(b)):
            continue
        nM = _prefix_for(b)
    elif CERTIFIED_ANY_MACHINE:
        # certified under this machine AND others, possibly at different
        # lengths. Take the longest, for the reason in _prefix_for.
        nM = _prefix_for(b) or nM
    terms = b.get("terms") or []
    nM = nM or 5
    if len(terms) < nM + K_ASK:
        continue
    ITEMS.append(dict(
        id=b["id"], cell=(b.get("want_retrieval"), b.get("want_transformation")),
        r_hat=b.get("r_hat"), t_hat=b.get("t_hat"),
        K=b.get(f"K_{MACHINE_TAG}"), n_M=nM,
        # the prefix is the one the item CERTIFIED at, so the examinee sees
        # exactly what the measure says is enough to fix the continuation
        x_pref=terms[:nM], truth=terms[nM:nM + K_ASK],
        h=((MET.get(b["id"]) or {}).get(MACHINE_ID) or {}).get("h")))

if ITEM_ORDER == "spread":
    ITEMS = spread_order(ITEMS)
elif ITEM_ORDER == "easiest":
    # LEAST EXPENSIVE FIRST, by h under the reporting machine, with items
    # that have no h at the end. For a model that may not finish, this puts
    # what it can answer at the front: a run stopped part way then covers the
    # low end of the scale completely instead of sampling all of it thinly,
    # and the items never reached are the expensive ones, where failure is
    # the likelier outcome anyway.
    ITEMS = sorted(ITEMS, key=lambda it: (it["h"] is None,
                                          it["h"] if it["h"] is not None else 0))
if MAX_ITEMS:
    ITEMS = ITEMS[:MAX_ITEMS]
log(f"  {len(ITEMS)} to administer"
    + (", certified only" if ONLY_CERTIFIED else "")
    + f", each shown its n_M terms and asked for {K_ASK} more")
_with_h = sum(1 for i in ITEMS if i["h"] is not None)
log(f"  {_with_h} of them have an h, so the report can relate score to "
    f"difficulty" if _with_h else
    f"  NONE has an h. The score columns still work; the h column will be "
    f"empty until the annotator has run and its metrics file is attached.")

# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
if EXAMINEE_PROBE and API_EXAMINEES:
    stage("CAN THE EXAMINEES ANSWER AT ALL?")
    log(f"  one trivial question each, before anything is spent. A retired or "
        f"misnamed model returns nothing, and a run of empty replies scores")
    log(f"  0% and looks like a result rather than a fault.\n")
    def _transient(note):
        n = str(note or "").lower()
        return any(w in n for w in ("529", "overloaded", "429", "rate limit",
                                    "rate_limit", "timeout", "timed out",
                                    "503", "unavailable", "500"))

    _dead = []
    for _gm, _b in pairs(API_EXAMINEES):
        _ans, _note = "", ""
        for _try in range(1, PROBE_RETRIES + 2):
            _txt, _note, _u = ask_any(
                "Continue the sequence with the next 3 terms only.\n"
                "Sequence: 1, 2, 3, 4\n"
                "Answer with the terms alone, separated by commas.",
                model=_gm, temperature=0.0, max_out=2000)
            _ans = last_line(_txt)[:60] if _txt else ""
            if _ans or not _transient(_note):
                break
            log(f"  wait  {_gm:26s} [{provider_of(_gm)}] -> busy, retrying in "
                f"{PROBE_BACKOFF_S}s ({_try} of {PROBE_RETRIES})")
            time.sleep(PROBE_BACKOFF_S)
        _ok = bool(_ans)
        log(f"  {'ok  ' if _ok else 'DEAD'}  {_gm:26s} [{provider_of(_gm)}] -> "
            + (repr(_ans) if _ok else f"nothing back ({_note})"))
        if not _ok:
            _dead.append((_gm, _note))

    # A model that is only BUSY does not stop the run. It is dropped for this
    # pass and its rows are picked up by a later one, since FILL_GAPS_ONLY
    # takes whatever is missing.
    _busy = [(m, n) for m, n in _dead if _transient(n)]
    _gone = [(m, n) for m, n in _dead if not _transient(n)]
    if _busy:
        _skip = {m for m, _ in _busy}
        API_EXAMINEES = [m for m in API_EXAMINEES if m not in _skip]
        log("")
        log(f"  {', '.join(sorted(_skip))} is at capacity after "
            f"{PROBE_RETRIES} retries, so it sits this pass out. Run again "
            f"with FILL_GAPS_ONLY to collect its rows; nothing else is held up.")
    if _gone:
        raise SystemExit(
            "stopping: " + ", ".join(m for m, _ in _gone) + " returned nothing "
            "and not because the provider is busy. Either the name is wrong or "
            "the model is no longer served. Check "
            "https://ai.google.dev/gemini-api/docs/deprecations, fix "
            "API_EXAMINEES, and run again. Nothing has been spent.")
    log(f"\n  all {len(pairs(API_EXAMINEES))} answered.")

# ---------------------------------------------------------------------------
stage("IS THE JUDGE SCORING?")
_bad = []
for seq, truth, resp, want in JUDGE_PROBES:
    got = score(seq.split(", "), truth.split(", "), resp)
    ok = got == want
    log(f"  {'ok  ' if ok else 'FAIL'}  {resp[:34]:36s} -> {got}, wanted {want}")
    if not ok: _bad.append((resp, got, want))
if _bad:
    raise SystemExit(
        f"the judge failed {len(_bad)} of {len(JUDGE_PROBES)} probes. Stopping: "
        f"a judge that cannot score these cannot score the battery, and the "
        f"numbers would look like results.")


## 9 · Run

In [ ]:
_API, _LOC = pairs(API_EXAMINEES), pairs(LOCAL_EXAMINEES)
stage(f"ADMINISTERING TO {len(_API) + len(_LOC)} EXAMINEE(S)")
if WALL_CLOCK_MIN:
    log(f"  stopping after {WALL_CLOCK_MIN} min whatever is left. What is "
        f"finished is kept and what was not reached is named.")
_left = lambda: (not WALL_CLOCK_MIN) or ((time.time()-T0)/60 < WALL_CLOCK_MIN)
_skipped = []
T0 = time.time()
RESULTS.clear()

# WHAT EACH EXAMINEE STILL OWES.
#
# An examinee that has answered an item does not need to answer it again, and
# reruns cost money and add nothing: at temperature 0 the reply is usually the
# same. So the previous results are read and each examinee is asked only for
# what is missing. A new examinee is missing everything and gets the full set;
# an examinee that has done all of them is skipped and said so.
_done = collections.defaultdict(set)
_prev_rows = []
if FILL_GAPS_ONLY:
    _prev_rows = []
    try:
        _prev_rows = json.load(open(find(RESULTS_FILE) or RESULTS_FILE))["rows"]
        for r in _prev_rows:
            _done[r["examinee"]].add(r["item"])
        log(f"\n  {len(_prev_rows)} rows already recorded, over "
            f"{len(_done)} examinee(s)")
    except Exception as e:
        log(f"\n  no previous results to read ({str(e)[:60]}), so every "
            f"examinee gets every item")

# STALE PREFIXES. A row records the n_M it was administered at. If a machine
# added since then wants a LONGER prefix, the item was questionable under that
# machine at the length used: a rival reproduces the shorter prefix and then
# diverges, so an examinee answering differently may not be wrong. Those rows
# cannot be used in an analysis relative to the newer machine and the item has
# to be asked again.
#
# The check is exact rather than a guess, because the row carries the number.
_at = {}
for r in _prev_rows:
    _at.setdefault(r["item"], set()).add(r.get("n_M"))
_stale = {it["id"]: (max(_at[it["id"]]), it["n_M"])
          for it in ITEMS
          if it["id"] in _at and _at[it["id"]]
          and max(_at[it["id"]]) < it["n_M"]}
if _stale:
    log(f"\n  {len(_stale)} item(s) were answered at a SHORTER prefix than the "
        f"battery now asks for, because a machine added since wants more:")
    for _i, (_was, _now) in sorted(_stale.items()):
        log(f"      {_i}: answered at n_M {_was}, now {_now}")
    log(f"  Those answers are not valid for an analysis relative to the "
        f"stricter machine, so the items are asked again below and the old "
        f"rows are kept")
    log(f"  with their own n_M, since they remain valid for the machine they "
        f"were collected under.")

def _todo_for(gm):
    if not FILL_GAPS_ONLY:
        return ITEMS
    return [it for it in ITEMS
            if it["id"] not in _done.get(gm, ()) or it["id"] in _stale]

for gm, _bud in _API:
    if not _left(): _skipped.append((gm, _bud)); continue
    _todo = _todo_for(gm)
    if not _todo:
        log(f"\n  {gm}   all {len(ITEMS)} items already answered, skipping")
        continue
    log(f"\n  {gm}   budget {_bud}   {len(_todo)} items x {REPS} reps"
        + (f"   ({len(ITEMS)-len(_todo)} already answered)"
           if len(_todo) < len(ITEMS) else ""))
    rows = administer(gm, _bud, items=_todo)
    RESULTS += rows
    _n = sum(r["n_correct"] for r in rows)
    _k = sum(r["k"] for r in rows)
    _empty = sum(1 for r in rows if not r["answer"])
    log(f"    {_n}/{_k} terms = {_n/max(_k,1):.1%}"
        + (f"   !! {_empty} EMPTY replies, so this score is not meaningful"
           if _empty else ""))
    save_results()

PROBE_LOCAL      = False  # probes 1 and 2 have run
PROBE_MAX_TOKENS = 1200
PROBE_ITEM       = "v17-0023"   # 1, 2, 3, 4, 5. If a model needs thousands of
                                # tokens for this one, it is not thinking.

if PROBE_LOCAL and LOCAL_EXAMINEES:
    _pname = LOCAL_EXAMINEES[0]
    log("")
    log("=" * 72)
    log(f"== PROBE 2: greedy against sampled, and the END of the reply")
    log("=" * 72)
    log("  The first probe printed only the start of each reply, which cannot")
    log("  tell long reasoning from a repetition loop. It also missed that")
    log("  generate() runs greedy, do_sample=False, while Phi-4-reasoning is")
    log("  documented for temperature 0.8 and top_p 0.95. Greedy decoding is")
    log("  a known way to put a reasoning model into a loop it never leaves,")
    log("  which would explain a reply that never closes </think>.")
    log("")
    _pm = Examinee(_pname)
    _it = next((x for x in ITEMS if x["id"] == PROBE_ITEM), None) or ITEMS[0]
    _prompt = EXAM_PROMPT.format(k=len(_it["truth"]),
                                 seq=", ".join(_it["x_pref"]))
    _msgs = [{"role": "user", "content": _prompt}]
    try:
        _text = _pm.tok.apply_chat_template(_msgs, tokenize=False,
                                            add_generation_prompt=True,
                                            enable_thinking=LOCAL_THINK)
    except TypeError:
        _text = _pm.tok.apply_chat_template(_msgs, tokenize=False,
                                            add_generation_prompt=True)
    log(f"  the prompt ends with: {_text[-90:]!r}\n")
    _enc = _pm.tok(_text, return_tensors="pt").to(_pm.model.device)
    import time as _t

    def _run(label, **kw):
        _t0 = _t.time()
        with _pm.torch.no_grad():
            _o = _pm.model.generate(
                **_enc, max_new_tokens=PROBE_MAX_TOKENS,
                eos_token_id=(_pm.eos_ids or _pm.tok.eos_token_id),
                pad_token_id=_pm.tok.pad_token_id, **kw)
        _new = _o[0][_enc["input_ids"].shape[1]:]
        _n = int(_new.shape[0])
        _txt = _pm.tok.decode(_new, skip_special_tokens=False)
        # a loop shows as one line repeated; count distinct 40-character
        # blocks against the total
        _bl = [_txt[i:i + 40] for i in range(0, len(_txt), 40)]
        _uniq = len(set(_bl)) / max(len(_bl), 1)
        log(f"  {label}: {_n} tokens in {_t.time()-_t0:.0f}s, "
            f"stopped early {_n < PROBE_MAX_TOKENS}, "
            f"closed </think> {'</think>' in _txt}, "
            f"distinct blocks {_uniq:.0%}")
        log(f"     START {_txt[:220]!r}")
        log(f"     END   {_txt[-220:]!r}")
        log("")

    _run("greedy (what the run does)", do_sample=False)
    _run("sampled, Phi-4's own settings", do_sample=True,
         temperature=0.8, top_p=0.95)

    _pm.unload()
    log("  READING IT")
    log("    sampled stops and greedy does not: greedy is the cause, and the")
    log("      run needs do_sample for this model.")
    log("    both run to the budget but distinct blocks is low: the model is")
    log("      looping whatever the sampling, and a repetition penalty is next.")
    log("    both run on with distinct blocks high: it really is writing that")
    log("      much, and no budget we can afford will fit it.")
    log("=" * 72)
    raise SystemExit("probe 2 finished; set PROBE_LOCAL = False to run for real")

for name, _bud in _LOC:
    if not _left(): _skipped.append((name, _bud)); continue
    log(f"\n  {name}   budget {_bud}   loading")
    m = Examinee(name)
    rows = administer(name, _bud, is_local=True, model=m,
                          items=_todo_for(name))
    m.unload()
    # NOT `RESULTS += rows` here. The local path adds them as it goes, so
    # that a session ending mid-examinee still leaves them on disk, and
    # adding them again would double every row.
    _n = sum(r["n_correct"] for r in rows)
    _k = sum(r["k"] for r in rows)
    log(f"    {_n}/{_k} terms = {_n/max(_k,1):.1%}")
    save_results()

_cut = sum(1 for r in RESULTS if r.get("was_cut"))
log(f"\n{len(RESULTS)} rows in {(time.time()-T0)/60:.1f} min"
    + (f", {_cut} answers cut at the budget" if _cut else ""))
if _skipped:
    log(f"  NOT REACHED, the {WALL_CLOCK_MIN} min limit came first: "
        + ", ".join(f"{n} at {b}" for n, b in _skipped))


## 10 · What it says

In [ ]:
def _short(name):
    """A short column label. The caption says what the letters are.

    gemini-3.5-flash-lite  ->  g3.5FL
    allenai/OLMo-2-1124-7B-Instruct  ->  O2-1124
    Qwen/Qwen2.5-1.5B-Instruct  ->  Q2.5-1.5

    Long names make a table of six examinees unreadable, and truncating them
    turned two different models into the same label.
    """
    import re
    n = str(name).split("/")[-1]
    m = re.match(r"gemini-([0-9.]+)-(.+)", n)
    if m:
        return "g" + m.group(1) + "".join(w[0].upper()
                                          for w in m.group(2).split("-"))
    m = re.match(r"OLMo-([0-9]+)-([0-9]+)", n, re.I)
    if m:
        return f"o{m.group(1)}-{m.group(2)}"
    m = re.match(r"Qwen([0-9.]+)-([0-9.]+)B", n, re.I)
    if m:
        return f"q{m.group(1)}-{m.group(2)}"
    m = re.match(r"gemma-([0-9]+)", n, re.I)
    if m:
        return f"gemma{m.group(1)}"
    return n[:12]

stage("WHAT THE EXAMINEES DID")
by_ex = collections.defaultdict(list)
for r in RESULTS: by_ex[r["examinee"]].append(r)

log(f"{'examinee':30s} {'score':>8s} {'items right':>12s}  {'reps agree':>10s}")
for ex, rows in by_ex.items():
    n = sum(r["n_correct"] for r in rows); k = sum(r["k"] for r in rows)
    per = collections.defaultdict(list)
    for r in rows: per[r["item"]].append(r["n_correct"])
    full = sum(1 for v in per.values() if st.mean(v) == rows[0]["k"])
    agree = sum(1 for v in per.values() if len(set(v)) == 1)
    log(f"{ex:30s} {n/max(k,1):7.1%} {full:6d}/{len(per):<5d} "
        f"{agree/max(len(per),1):9.0%}")

# what each examinee SPENT, and whether the budget bound it. A score is not
# readable without this: an examinee that hit the cap on a third of the items
# was not answering those items, it was running out of room.
log(f"\nwhat each answer cost, and whether the budget bound it")
log(f"{'examinee':22s} {'budget':>7s} {'median':>7s} {'p90':>7s} "
    f"{'think':>6s} {'cut':>9s} {'looped':>7s}")
for ex, rows in by_ex.items():
    _t = sorted(r.get("tok_out") or 0 for r in rows)
    _cut = sum(1 for r in rows if r.get("was_cut"))
    _lp = sum(1 for r in rows if r.get("looked_looped"))
    _sh = [r["think_share"] for r in rows if r.get("think_share") is not None]
    _b = rows[0].get("max_out")
    if not _t: continue
    log(f"{_short(ex):22s} {str(_b):>7s} {_t[len(_t)//2]:7d} "
        f"{_t[int(.9*len(_t))]:7d} "
        + (f"{st.mean(_sh):5.0%} " if _sh else f"{'-':>6s}")
        + f"{_cut:4d}/{len(rows):<4d} {_lp:7d}")
log(f"  `cut' is measured, the model was stopped before its end token. "
    f"`looped' is a guess from the tail of a cut answer and is NOT the reason")
log(f"  for the cut: a model cut off may have been repeating itself or may "
    f"have been working steadily and run out of room.")
_bad = [(sum(1 for r in rows if r.get("was_cut"))/len(rows), ex)
        for ex, rows in by_ex.items()]
_w = max(_bad) if _bad else (0, None)
if _w[0] > 0.1:
    log(f"  {_short(_w[1])} was cut on {_w[0]:.0%} of its answers. Its score is "
        f"a budget, not an ability; raise the budget for that pair.")
log(f"  `thinking' is the share of the output spent reasoning before the "
    f"answer. It is what a compute-against-accuracy reading needs, and it is")
log(f"  recorded per answer, so accuracy can be read against tokens spent as "
    f"well as against the cost of the item.")

def _corr(a, b):
    n = len(a)
    if n < 3: return float("nan")
    ma, mb = sum(a)/n, sum(b)/n
    da = math.sqrt(sum((x-ma)**2 for x in a))
    db = math.sqrt(sum((y-mb)**2 for y in b))
    return sum((x-ma)*(y-mb) for x, y in zip(a, b))/(da*db) if da*db else 0.0

log(f"\nscore against the measures, per examinee. A difficulty scale should "
    f"give a NEGATIVE correlation:")
log(f"{'examinee':30s} {'vs h':>8s} {'vs K':>8s} {'vs r_hat':>9s} {'vs t_hat':>9s}")
for ex, rows in by_ex.items():
    per = collections.defaultdict(list)
    for r in rows: per[r["item"]].append(r)
    acc, hs, ks, rs, ts = [], [], [], [], []
    for iid, rr in per.items():
        a = st.mean(r["n_correct"] / r["k"] for r in rr)
        acc.append(a); hs.append(rr[0]["h"]); ks.append(rr[0]["K"])
        rs.append(rr[0]["r_hat"]); ts.append(rr[0]["t_hat"])
    ph = [(a, h) for a, h in zip(acc, hs) if h is not None]
    pk = [(a, x) for a, x in zip(acc, ks) if x is not None]
    log(f"{ex:30s} "
        + (f"{_corr([a for a,_ in ph], [h for _,h in ph]):+8.2f}" if len(ph) > 2 else f"{'-':>8s}")
        + (f"{_corr([a for a,_ in pk], [x for _,x in pk]):+8.2f}" if len(pk) > 2 else f"{'-':>8s}")
        + f"{_corr(rs, acc):+9.2f}{_corr(ts, acc):+9.2f}")
log(f"  h is on {sum(1 for i in ITEMS if i['h'] is not None)} of "
    f"{len(ITEMS)} items, so the h column rests on those only.")

hard = sorted({r["item"] for r in RESULTS},
              key=lambda i: st.mean(r["n_correct"]/r["k"] for r in RESULTS
                                    if r["item"] == i))
log(f"\nhardest for everyone:")
for iid in hard[:5]:
    rr = [r for r in RESULTS if r["item"] == iid]
    log(f"   {iid}  r{rr[0]['r_hat']} t{rr[0]['t_hat']}  K {rr[0]['K']}  "
        f"{st.mean(r['n_correct']/r['k'] for r in rr):.0%}")
log(f"easiest:")
for iid in hard[-5:]:
    rr = [r for r in RESULTS if r["item"] == iid]
    log(f"   {iid}  r{rr[0]['r_hat']} t{rr[0]['t_hat']}  K {rr[0]['K']}  "
        f"{st.mean(r['n_correct']/r['k'] for r in rr):.0%}")


## 11 · Save

In [ ]:
save_results()
log(f"wrote {RESULTS_FILE}: {len(RESULTS)} rows")

account(units=len(RESULTS), note=f"evaluated {len(set(r['examinee'] for r in RESULTS))} "
        f"examinee(s) on {len(ITEMS)} items")

if PUSH_RESULTS:
    state_save(f"{BUILD}: {len(RESULTS)} result rows")
    log("pushed to the dataset")
else:
    log(f"\n  ###############################################################")
    log(f"  #  NOT PUSHED. {RESULTS_FILE} exists in this session only.")
    log(f"  #  DOWNLOAD IT NOW. When the session ends it is gone.")
    log(f"  #")
    log(f"  #  It was not pushed because a push replaces the whole dataset")
    log(f"  #  and stages files this run only read, so it would undo an")
    log(f"  #  annotation run going on elsewhere. Set PUSH_RESULTS = True")
    log(f"  #  when nothing else is running.")
    log(f"  ###############################################################")
